# Full RAG Evaluation — Retrieval + Generation, on Real Data

```
                     RAG Evaluation
                          |
             +-------------------+-------------------+
             |                                        |
             v                                        v
        RETRIEVAL                               GENERATION
             |                                        |
    +--------+--------+                    +----------+----------+
    v        v        v                    v          v          v
 Recall   Precision  Hit Rate         Faithfulness  Relevancy  Correctness
    |        |        |                    |          |          |
    +--------+--------+                    +----------+----------+
             v                                        v
          Ranking                              Answer Quality
             |
       +-----+-----+
       v     v     v
      MRR   MAP   NDCG
```

`l27_ragas_evaluation_dataset.ipynb` generated a synthetic dataset (`data/ragas_eval_dataset.jsonl`) of questions with ground-truth `reference_contexts` and `reference` answers. `l28_ragas_retrieval_metrics.ipynb` and `l29_retrieval_metrics_tutorial.ipynb` covered pieces of this diagram separately (Context Precision/Recall, and the metric formulas on a toy example). This notebook computes **every leaf of the diagram above, on the real dataset**, in one place:

- **Retrieval branch** (Recall, Precision, Hit Rate, MRR, MAP, NDCG): Ragas has no built-in classes for these classic rank-based metrics (its retrieval metrics are the LLM-judged Context Precision/Recall from `l28`, not these). So this notebook computes them directly, reusing the exact formulas from `l29_retrieval_metrics_tutorial.ipynb`, applied to real retrieved chunks compared against each question's real `reference_contexts`.
- **Generation branch** (Faithfulness, Relevancy, Correctness): these *are* real Ragas metrics (`Faithfulness`, `AnswerRelevancy`, `AnswerCorrectness`), used directly.

**Run `l27_ragas_evaluation_dataset.ipynb` first** if `data/ragas_eval_dataset.jsonl` doesn't exist yet.

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

SCHEMES_PDF = "data/sampled_schemes.pdf"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Generates the actual RAG answers below -- plain single-response calls at the model's
# default temperature, which gpt-5.6-luna supports fine (only *custom* temperatures fail).
llm = ChatOpenAI(model="gpt-5.6-luna")

# Ragas' generation metrics request custom temperatures internally (same issue as l27/l28),
# so metric judging uses a temperature-flexible model instead.
ragas_llm = ChatOpenAI(model="gpt-5.4-nano")

In [3]:
import sys
import types

# Create a fake module to satisfy ragas's broken hardcoded import
fake_module = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:
    def __init__(self, *args, **kwargs):
        raise ImportError("ChatVertexAI is not actually available — install langchain-google-vertexai if you need it.")

fake_module.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = fake_module

## 1. Load the evaluation dataset

In [4]:
from ragas.testset.synthesizers.testset_schema import Testset

testset = Testset.from_jsonl("data/ragas_eval_dataset.jsonl")
eval_dataset = testset.to_evaluation_dataset()

print(f"{len(eval_dataset.samples)} evaluation samples")
eval_dataset.samples[0]

10 evaluation samples


SingleTurnSample(user_input='What is NATIONAL FOOD SECURITY MISSION?', retrieved_contexts=None, reference_contexts=['## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL FOOD SECURITY MISSION'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='The context only mentions NATIONAL FOOD SECURITY MISSION under the heading CENTRALLY SPONSORED SCHEMES, but it does not provide any additional details about what it is.', rubrics=None, persona_name='MarÃ\xada Ortega', query_style='POOR_GRAMMAR', query_length='SHORT')

## 2. Rebuild the retrieval pipeline

Same Docling + `MarkdownTextSplitter` chunking as `l27_ragas_evaluation_dataset.ipynb` -- **the chunk size/overlap here must match what produced `reference_contexts`** (`600`/`120`), since the retrieval metrics below match retrieved chunks against `reference_contexts` by exact text, not just semantic similarity.

In [5]:
os.environ["TORCHDYNAMO_DISABLE"] = "1"
logging.getLogger("docling").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(do_ocr=False)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

schemes_doc = converter.convert(SCHEMES_PDF).document
schemes_md = schemes_doc.export_to_markdown()

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 8649.28it/s]


In [6]:
from langchain_text_splitters import MarkdownTextSplitter

splitter = MarkdownTextSplitter(chunk_size=600, chunk_overlap=120)
chunks = splitter.create_documents([schemes_md], metadatas=[{"source": SCHEMES_PDF}])

print(f"{len(chunks)} chunks")

39 chunks


In [7]:
import shutil
from pathlib import Path
from langchain_chroma import Chroma

CHROMA_DIR = Path("vector_stores/rag_eval_chroma_db")
shutil.rmtree(CHROMA_DIR, ignore_errors=True)

chroma_store = Chroma(
    collection_name="rag_eval_schemes",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)
chroma_store.add_documents(chunks)
retriever = chroma_store.as_retriever(search_kwargs={"k": 5})

## 3. Retrieval branch: Recall, Precision, Hit Rate, MRR, MAP, NDCG

Same formulas as `l29_retrieval_metrics_tutorial.ipynb`, just applied here for real: for each question, "relevant chunks" = the text in that question's `reference_contexts`, and the retriever's top-5 results (in rank order) are compared against that set. There's no graded relevance label for real chunks (unlike the toy `[3, 0, 1, 3]` example in `l29`), so NDCG here uses binary relevance (1 = matches a reference chunk, 0 = doesn't) -- still a valid, commonly-used form of NDCG, just without partial-credit grading.

In [8]:
import math

def precision_at_k(hits):
    return sum(hits) / len(hits)

def recall_at_k(hits, n_relevant):
    return sum(hits) / n_relevant if n_relevant else 0.0

def hit_at_k(hits):
    return 1 if any(hits) else 0

def reciprocal_rank(hits):
    for rank, hit in enumerate(hits, start=1):
        if hit:
            return 1 / rank
    return 0.0

def average_precision(hits, n_relevant):
    hit_count, precision_sum = 0, 0.0
    for rank, hit in enumerate(hits, start=1):
        if hit:
            hit_count += 1
            precision_sum += hit_count / rank
    return precision_sum / n_relevant if n_relevant else 0.0

def ndcg(hits):
    dcg = sum(hit / math.log2(i + 1) for i, hit in enumerate(hits, start=1))
    ideal_hits = sorted(hits, reverse=True)
    idcg = sum(hit / math.log2(i + 1) for i, hit in enumerate(ideal_hits, start=1))
    return dcg / idcg if idcg > 0 else 0.0

In [9]:
retrieval_rows = []

for sample in eval_dataset.samples:
    retrieved_docs = retriever.invoke(sample.user_input)
    sample.retrieved_contexts = [doc.page_content for doc in retrieved_docs]  # also needed for generation metrics later

    relevant_texts = set(sample.reference_contexts)
    hits = [1 if text in relevant_texts else 0 for text in sample.retrieved_contexts]
    n_relevant = len(relevant_texts)

    retrieval_rows.append({
        "user_input": sample.user_input,
        "precision_at_5": precision_at_k(hits),
        "recall_at_5": recall_at_k(hits, n_relevant),
        "hit_at_5": hit_at_k(hits),
        "reciprocal_rank": reciprocal_rank(hits),
        "average_precision": average_precision(hits, n_relevant),
        "ndcg": ndcg(hits),
    })

import pandas as pd
retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df

,user_input,precision_at_5,recall_at_5,hit_at_5,reciprocal_rank,average_precision,ndcg
0,What is NATIONAL FOOD SECURITY MISSION?,0.2,1.0,1,1.0,1.0,1.0
1,"How was the scheme implemented from 2007-08, a...",0.0,0.0,0,0.0,0.0,0.0
2,Prices ka kya matlab hai aur ye kaise under ch...,0.0,0.0,0,0.0,0.0,0.0
3,Farm machineries distribution under NMSA and w...,0.0,0.0,0,0.0,0.0,0.0
4,What is the purpose of setting up project mana...,0.0,0.0,0,0.0,0.0,0.0
5,How is the NMSA scheme implemented through the...,0.0,0.0,0,0.0,0.0,0.0
6,What centrally sponsored scheme mentioned in t...,0.0,0.0,0,0.0,0.0,0.0
7,In the context of cluster demonstrations and s...,0.0,0.0,0,0.0,0.0,0.0
8,What does the context say about using ICT in a...,0.0,0.0,0,0.0,0.0,0.0
9,"Under NMSA, how does the PACS-linked process h...",0.0,0.0,0,0.0,0.0,0.0


In [10]:
retrieval_summary = retrieval_df.drop(columns="user_input").mean().rename({
    "precision_at_5": "Precision@5",
    "recall_at_5": "Recall@5",
    "hit_at_5": "Hit Rate@5",
    "reciprocal_rank": "MRR",
    "average_precision": "MAP@5",
    "ndcg": "NDCG@5",
})
retrieval_summary

Precision@5    0.02
Recall@5       0.10
Hit Rate@5     0.10
MRR            0.10
MAP@5          0.10
NDCG@5         0.10
dtype: float64

## 4. Generate answers

A minimal RAG prompt: retrieved chunks (already populated above) + the question, answered by `llm`.

In [11]:
RAG_PROMPT = """Answer the question using only the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}"""

for sample in eval_dataset.samples:
    context = "\n\n".join(sample.retrieved_contexts)
    prompt = RAG_PROMPT.format(context=context, question=sample.user_input)
    sample.response = llm.invoke(prompt).content

print(eval_dataset.samples[0].user_input)
print(eval_dataset.samples[0].response)

What is NATIONAL FOOD SECURITY MISSION?
The **National Food Security Mission (NFSM)** is a Central Sector Scheme implemented since **2007–08** to increase the production and productivity of **rice and pulses** in selected districts. It also covers **coarse cereals, commercial crops, nutri-cereals and oilseeds**, including pulses and oilseeds in rice-fallow areas.

Its objectives are to strengthen food security, increase government food stocks, maintain self-sufficiency and keep prices under control.


## 5. Generation branch: Faithfulness, Relevancy, Correctness

- **Faithfulness** — does the generated answer only make claims that are actually supported by the retrieved context (checks for hallucination)?
- **Relevancy** (`AnswerRelevancy`) — does the answer actually address the question asked (independent of whether it's correct or grounded)?
- **Correctness** (`AnswerCorrectness`) — does the answer match the ground-truth `reference`, combining factual overlap and semantic similarity?

These are real Ragas metric classes (unlike the retrieval branch above). Same `ragas.metrics` legacy-but-functional import path as `l28_ragas_retrieval_metrics.ipynb`, for the same reasons.

In [12]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness

generation_result = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness(), AnswerRelevancy(), AnswerCorrectness()],
    llm=ragas_llm,
    embeddings=embeddings,
)

generation_df = generation_result.to_pandas()
generation_df

Evaluating:  13%|█▎        | 4/30 [00:09<00:46,  1.79s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 30/30 [00:35<00:00,  1.17s/it]


,user_input,retrieved_contexts,reference_contexts,response,reference,persona_name,query_style,query_length,faithfulness,answer_relevancy,answer_correctness
0,What is NATIONAL FOOD SECURITY MISSION?,[## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL...,[## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL...,The **National Food Security Mission (NFSM)** ...,The context only mentions NATIONAL FOOD SECURI...,MarÃ­a Ortega,POOR_GRAMMAR,SHORT,1.00,0.839362,0.165039
1,"How was the scheme implemented from 2007-08, a...",[The scheme is being implemented from 2007 ­ 0...,[The scheme is being implemented from 2007 Â­ ...,The scheme was implemented from 2007–08 as a *...,The scheme is being implemented from 2007-08 a...,Dr. Kavya Menon,PERFECT_GRAMMAR,SHORT,1.00,0.706688,0.979197
2,Prices ka kya matlab hai aur ye kaise under ch...,[facilities in India is regulatory in nature w...,[Objectives: Increase food security by steppin...,दिए गए संदर्भ में “prices” का स्पष्ट अर्थ या उ...,"Context ke mutabiq, objectives me ""prices rema...",Dr. Kavya Menon,POOR_GRAMMAR,SHORT,0.80,0.413993,0.263018
3,Farm machineries distribution under NMSA and w...,[Interventions: Provision of subsidy to farmer...,[- ïƒ¼ Cluster Demonstrations - test beds for ...,The context does not specify farm machinery di...,Farm Machineries refers to the distribution of...,Dr. Kavya Menon,MISSPELLED,SHORT,1.00,0.691743,0.856675
4,What is the purpose of setting up project mana...,[-  Plant Protection - Subsidised distributio...,[- ïƒ¼ Plant Protection - Subsidised distribut...,The context does not specify the purpose of se...,The context states that part of the program in...,MarÃ­a Ortega,PERFECT_GRAMMAR,SHORT,1.00,0.000000,0.199711
5,How is the NMSA scheme implemented through the...,[## NATIONAL MISSION FOR SUSTAINABLE AGRICULTU...,[## NATIONAL MISSION FOR SUSTAINABLE AGRICULTU...,The NMSA scheme is implemented through the Dir...,The NMSA scheme is being implemented with diff...,Asha Verma,WEB_SEARCH_LIKE,SHORT,1.00,0.734076,0.771552
6,What centrally sponsored scheme mentioned in t...,[## CENTRALLY SPONSORED SCHEMES\n\n## NATIONAL...,[<1-hop>\n\n## CENTRALLY SPONSORED SCHEMES\n\n...,The centrally sponsored scheme mentioned is th...,The centrally sponsored scheme mentioned in th...,NaN,NaN,NaN,1.00,0.810204,0.972259
7,In the context of cluster demonstrations and s...,[-  Cluster Demonstrations - test beds for de...,[<1-hop>\n\n- ïƒ¼ Cluster Demonstrations - tes...,The subsidized farm machinery distribution sup...,The context explains that cluster demonstratio...,NaN,NaN,NaN,0.50,0.813850,0.817840
8,What does the context say about using ICT in a...,"[""Make all government services accessible to t...",[<1-hop>\n\n## Interventions:\n\n- ïƒ¼ Soil He...,The context presents ICT as a means to make go...,The context presents ICT in agriculture and go...,NaN,NaN,NaN,0.85,0.879432,0.575099
9,"Under NMSA, how does the PACS-linked process h...",[## 3.1.1 Compulsory Component\n\n- 3.1.1.1 Al...,[<1-hop>\n\n## 3.1.1 Compulsory Component\n\n-...,"Under the compulsory component, every farmer w...",Compulsory coverage applies to all farmers who...,NaN,NaN,NaN,0.80,0.770054,0.806043


## 6. Save results

In [13]:
retrieval_df.to_csv("data/rag_retrieval_ranking_metrics.csv", index=False)
generation_df.to_csv("data/ragas_generation_metrics.csv", index=False)

print("saved data/rag_retrieval_ranking_metrics.csv and data/ragas_generation_metrics.csv")

saved data/rag_retrieval_ranking_metrics.csv and data/ragas_generation_metrics.csv


## Summary

Every leaf metric in this notebook's opening diagram is now computed against the real `data/sampled_schemes.pdf` pipeline: the retrieval branch (`retrieval_summary` below) from exact chunk-rank matching against `reference_contexts`, and the generation branch (`generation_df` above) from Ragas judging the actual generated `response` against `reference`.

In [15]:
import pandas as pd

numeric_cols = [c for c in generation_df.columns if pd.api.types.is_numeric_dtype(generation_df[c])]

print("RETRIEVAL")
print(retrieval_summary.round(3).to_string())
print("\nGENERATION")
print(generation_df[numeric_cols].mean().round(3).to_string())

RETRIEVAL
Precision@5    0.02
Recall@5       0.10
Hit Rate@5     0.10
MRR            0.10
MAP@5          0.10
NDCG@5         0.10

GENERATION
faithfulness          0.895
answer_relevancy      0.666
answer_correctness    0.641
